## Library installation & import

In [ ]:
!pip install numpy xarray cubo matplotlib rioxarray rasterio fiona utm pandas tensorflow

In [ ]:
!uv pip install -q --system numba-cuda==0.4.0

In [ ]:
import xarray as xr
import matplotlib.pyplot as plot
import matplotlib.patches as patches
import numpy as np
import rasterio
import fiona
import pandas as pd
import tensorflow as tf

from rasterio import mask
from tensorflow.keras import layers, Model, Input
import utm
import numba

from numba import jit
from numba import cuda
from numba import config
config.CUDA_ENABLE_PYNVJITLINK = 1
config.CUDA_LOW_OCCUPANCY_WARNINGS = 0

import subprocess

## Initialization

### Filesystem setup

Google drive connection

In [ ]:
from google.colab import drive
drive.mount('/content/drive/', force_remount=True)

Mounted at /content/drive/


Project directory setup

In [ ]:
HOME_DIR = '/content/drive/MyDrive/Predict Fire to Save Lives/Journal/submitted_script/'

### GPU check

In [ ]:
print(tf.__version__)
print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))

2.19.0
Num GPUs Available:  1


In [ ]:
try:
    subprocess.check_output('nvidia-smi')
    is_gpu_avail = True
except Exception: # this command not being found can raise quite a few different errors depending on the configuration
    is_gpu_avail = False

Files location

In [ ]:
model_file = HOME_DIR + 'neuralnet_model.keras'
input_dataset_file = HOME_DIR + 'dataset_bands.tif'
predicted_fuel_file = HOME_DIR + 'predicted_fuel_type.tif'

In [ ]:
ndvi_file = HOME_DIR + 'ndvi.tif'
ndmi_file = HOME_DIR + 'ndmi.tif'
fuel_type_file = predicted_fuel_file

In [ ]:
bayes_likelihood_file = HOME_DIR + "bayesian_likelihood.h5"

## Neural-net fuel type classification

### Load model

Open weight and biases model

In [ ]:
model = tf.keras.models.load_model(model_file)

Evaluate model

In [ ]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 64)             │           448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 4)              │           260 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 14,606 (57.06 KB)

 Trainable params: 4,868 (19.02 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 9,738 (38.04 KB)

### Fuel type prediction

Open input raster file to do prediction

In [ ]:
with rasterio.open(input_dataset_file) as rst:
  data_images = rst.read()
  out_meta = rst.meta

Add Softmax layer

In [ ]:
fuel_probability_model = tf.keras.Sequential([model, layers.Softmax()])

Reshape the raster file shape from (Bands, Y, X) to (Y * X, Bands), so it conformant to the input layer of Neural Network.

In [ ]:
prediction_dataset = np.zeros((data_images.shape[1] * data_images.shape[2], data_images.shape[0]))

for y in range(data_images.shape[1]):
  for x in range(data_images.shape[2]):
    count = (y * data_images.shape[1]) + x
    prediction_dataset[count] = data_images[:, y, x]

Create input tensor from the reshaped raster file

In [ ]:
prediction_tensor = tf.data.Dataset.from_tensor_slices((prediction_dataset))
prediction_tensor = prediction_tensor.batch(64)
prediction_tensor = prediction_tensor.prefetch(64)

Doing prediction

In [ ]:
prediction = fuel_probability_model.predict(prediction_tensor)

16529/16529 ━━━━━━━━━━━━━━━━━━━━ 21s 1ms/step


Based on softmax class probability, decide the class using argmax.

In [ ]:
prediction_result = np.zeros(prediction.shape[0])

for i, p in enumerate(prediction):
  pred = np.argmax(p)
  prediction_result[i] = pred

Reshape the result and do cloud masking.

In [ ]:
pred_final = prediction_result.reshape((data_images.shape[1], data_images.shape[2]))

cloud_mask = np.ma.masked_invalid(data_images[0])

pred_final = np.ma.masked_array(pred_final, cloud_mask.mask)

### Save fuel type prediction

Save the final fuel type prediction result

In [ ]:
kwargs = out_meta.copy()
kwargs.update(
    {
        'count': 1
    }
)

with rasterio.open(predicted_fuel_file, 'w', **kwargs) as dest:
    dest.write(pred_final, 1)

## Fire probability calculation

In [ ]:
def read_index(filename):
  with rasterio.open(filename, 'r') as rst:
    dt = rst.read(1)
    meta = rst.meta
    img = np.ma.masked_where(dt == 1e+20, dt)
    img = np.ma.masked_invalid(img)

  return img, meta

In [ ]:
def pixel_processing(indexes, likelihood_prob_table, prior, result):
  index_width = indexes.sizes['x']
  index_height = indexes.sizes['y']
  for y in range(index_height):
      for x in range(index_width):
          pixel_vals = indexes.isel(y=y, x=x)
          dry_pixel = pixel_vals.sel(index='dry').values.round(1)
          ndmi_pixel = pixel_vals.sel(index='ndmi').values.round(1)
          fuel_type_pixel = pixel_vals.sel(index='fuel_type').values.round(1)

          print(x, y)
          print(dry_pixel, ndmi_pixel, "\n")
          if np.isnan(dry_pixel) or np.isnan(ndmi_pixel) or np.isnan(fuel_type_pixel):
              result[y, x] = np.nan
              continue

          try:
              likelihood = likelihood_prob_table.sel(burned=True, dry=dry_pixel, ndmi=ndmi_pixel, fuel_type=fuel_type_pixel).values
              result[y, x] = likelihood
          except KeyError:
              result[y, x] = np.nan

In [ ]:
@cuda.jit
def pixel_processing_cuda(image_array, likelihood_table, dims, burned_vals, ndvi_vals, ndmi_vals, fuel_type_vals, result):
    j, i = cuda.grid(2)

    if j < result.shape[0] and i < result.shape[1]:
        image_index = j * result.shape[1] + i
        tmp = 0.0
        for val_idx, val in enumerate(image_array[image_index]):
            if dims[val_idx + 1] == 'ndvi':
                ndvi_coord = 0
                for ids, da in enumerate(ndvi_vals):
                    if da == val:
                        ndvi_coord = ids
                        break

            elif dims[val_idx + 1] == 'ndmi':
                ndmi_coord = 0
                for ids, da in enumerate(ndmi_vals):
                    if da == val:
                        ndmi_coord = ids
                        break

            elif dims[val_idx + 1] == 'fuel_type':
                fuel_type_coord = 0
                for ids, da in enumerate(fuel_type_vals):
                    if da == val:
                        fuel_type_coord = ids
                        break

        result[j][i] = likelihood_table[0][ndvi_coord][ndmi_coord][fuel_type_coord]

In [ ]:
def bayes_predict(prob_table, indicies, prior):
  indexes = indicies
  index_width = indexes.sizes['x']
  index_height = indexes.sizes['y']

  post_img = np.zeros((index_height, index_width))

  flatten_indexes = np.array([ indexes.values[:, y, x] for y in range(post_img.shape[0]) for x in range(post_img.shape[1])]).round(decimals=1)

  # likelihood_prob_table = xr.open_dataarray(HOME_DIR + "generated_data/bayesian_likelihood.h5")

  if not is_gpu_avail:
    pixel_processing(indexes, prob_table, prior, post_img)
  else:
    dims = cuda.to_device(prob_table.dims)
    burned_vals = cuda.to_device(prob_table.burned)
    ndvi_vals = cuda.to_device(prob_table.ndvi)
    ndmi_vals = cuda.to_device(prob_table.ndmi)
    fuel_type_vals = cuda.to_device(prob_table.fuel_type)
    likelihood_table_vals = cuda.to_device(prob_table)
    flatten_indexes_cuda = cuda.to_device(flatten_indexes)
    post_img_cuda = cuda.device_array_like(post_img)

    pixel_processing_cuda[(32, 32), (32, 32)](flatten_indexes_cuda, likelihood_table_vals, dims, burned_vals, ndvi_vals, ndmi_vals, fuel_type_vals, post_img_cuda)
    post_img = post_img_cuda.copy_to_host()

  posterior_img = post_img * prior / ((post_img * prior) + ((1 - post_img) * (1 - prior)))

  return posterior_img

In [ ]:
ndvi_raster, ndvi_raster_meta = read_index(ndvi_file)
ndmi_raster, ndmi_raster_meta = read_index(ndmi_file)
fuel_type_raster, fuel_type_raster_meta = read_index(fuel_type_file)

ndvi = ndvi_raster[:1000, :1000]
ndmi = ndmi_raster[:1000, :1000]
fuel_type = fuel_type_raster[:1000, :1000]

In [ ]:
prior = 0.5919471011925848 # based on learning on bromo 2023 data

In [ ]:
likelihood_prob_table = xr.open_dataarray(bayes_likelihood_file)

ndvi_ind = np.round(ndvi, decimals=1)
ndmi_ind = np.round(ndmi, decimals=1)
fuel_type_ind = np.round(fuel_type, decimals=1)

data_indicies = xr.DataArray(np.array([ndvi_ind, ndmi_ind, fuel_type_ind]), dims=["index", "y", "x"], coords={"index": ["ndvi", "ndmi", "fuel_type"]})

predicted = bayes_predict(likelihood_prob_table, data_indicies, prior)

nvJitLinkError: ERROR_INTERNAL (6)
Linker error log: ERROR 4 in nvvmAddNVVMContainerToProgram, may need newer version of nvJitLink library
 

Visualization

In [ ]:
fig, axes = plot.subplots((1, 4), figsize=(45, 10))

ax = axes[0]
ax.imshow(ndvi_ind)
ax.set_title('NDVI')

ax = axes[1]
ax.imshow(ndvi_ind)
ax.set_title('NDMI')

ax = axes[2]
ax.imshow(ndvi_ind)
ax.set_title('Predicted Fuel Type')

ax = axes[3]
ax.imshow(ndvi_ind)
ax.set_title('Fire Probability')

plot.show()